# Embedding cache comparison

Load and compare the window-embedding caches produced by `embed_windows.py`
(the `run_window_sweep.sh` sweep over half-window × snp-only). Provides:

1. Discovery + lazy loading of any cache in `checkpoints/sweep/`.
2. Per-window embedding norms (and **reference-delta** norms: how far each
   variant window departs from its variant-free reference, via
   `FPRefDeltaSumHeadModel`).
3. A cross-cache summary table.
4. Overlaid norm distributions.
5. Pairwise diff of two caches on their shared windows (align by fingerprint).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# This notebook lives in train_pipeline/; put the repo root on the path.
HERE = Path.cwd()
ROOT = HERE if (HERE / "crop_embed").exists() else HERE.parent
sys.path.insert(0, str(ROOT))

from crop_embed.models.fp_head_model import FPRefDeltaSumHeadModel

CACHE_DIR = ROOT / "checkpoints" / "sweep"   # <- point elsewhere to compare other caches

# Discover caches, skipping the *.embtable.ckpt.pt resume checkpoints.
cache_paths = sorted(
    p for p in CACHE_DIR.glob("*.ckpt.pt") if not p.name.endswith(".embtable.ckpt.pt")
)
print(f"Found {len(cache_paths)} caches in {CACHE_DIR}:")
for p in cache_paths:
    print("  ", p.name)

## 1. Load a cache

`load_cache` reads one cache lazily and caches it in `LOADED` so re-running
cells is cheap. Each entry exposes the `cache` table `(n_fps, D)`, the
fingerprint list (cache-row order), the sample→fingerprint index, and the
generation `metadata`.

In [ ]:
LOADED: dict[str, dict] = {}

def cache_name(path) -> str:
    # sativas413_hw250_snponly.ckpt.pt -> hw250_snponly
    stem = Path(path).name.replace(".ckpt.pt", "")
    return stem.replace("sativas413_", "")

def load_cache(path) -> dict:
    """Load one cache (memoized). Returns a dict of tensors + metadata."""
    name = cache_name(path)
    if name in LOADED:
        return LOADED[name]
    blob = torch.load(path, map_location="cpu", weights_only=False)
    fps = [(c, s, e, tuple(a)) for (c, s, e, a) in blob["unique_fingerprints"]]
    meta = blob.get("metadata", {}) or {}
    cc = {
        "name": name,
        "path": str(path),
        "cache": blob["cache"].float(),            # (n_fps, D)
        "fps": fps,                                # cache-row order
        "sample_fp_index": blob["sample_fp_index"].long(),
        "metadata": meta,
        "sample_ids": blob.get("sample_ids"),
        "half_window": meta.get("half_window"),
        "snp_only": meta.get("snp_only"),
        "attn_impl": meta.get("attn_impl"),
    }
    LOADED[name] = cc
    return cc

def load_all() -> dict[str, dict]:
    return {cache_name(p): load_cache(p) for p in cache_paths}

# Load everything up front (6 caches; a few hundred MB each).
caches = load_all()
print("Loaded:", list(caches))

## 2. Per-window norms and reference-delta norms

Each cache row is one unique fingerprint (a window + a specific set of alt
alleles). `cache_stats` computes:

- **norm**: L2 norm of each fingerprint embedding.
- **delta_norm**: L2 norm of `emb - emb_reference`, where the reference is the
  variant-free window `(chrom, w_start, w_end, ())`. This is exactly the
  quantity `FPRefDeltaSumHeadModel` pools — how much a variant window moves the
  embedding off its baseline. Reference rows have delta_norm == 0.

In [ ]:
def cache_stats(cc: dict) -> dict:
    cache = cc["cache"]
    norms = cache.norm(dim=1)                                   # (n_fps,)
    # strict=False: any window without a reference maps to itself (delta 0).
    ref_index = FPRefDeltaSumHeadModel.build_ref_index(cc["fps"], strict=False)
    delta_norms = (cache - cache[ref_index]).norm(dim=1)        # (n_fps,)
    is_ref = torch.tensor([len(a) == 0 for (_, _, _, a) in cc["fps"]])
    has_ref = ref_index != torch.arange(len(cc["fps"]))         # ref found (not self)
    # A row whose window's reference exists; reference rows themselves count as having one.
    has_ref = has_ref | is_ref
    return {
        "norms": norms,
        "delta_norms": delta_norms,
        "is_ref": is_ref,
        "has_ref": has_ref,
        "ref_index": ref_index,
    }

STATS = {name: cache_stats(cc) for name, cc in caches.items()}

Inspect one cache. Change `SELECT` to any discovered name (e.g. `hw250`, `hw500_snponly`, `hw1000`).

In [ ]:
SELECT = "hw250"   # <- pick any key from `caches`

cc = caches[SELECT]
st = STATS[SELECT]
variant = ~st["is_ref"]   # exclude reference rows from the delta view

print(f"{SELECT}: {cc['cache'].shape[0]:,} fingerprints x {cc['cache'].shape[1]} dims")
print(f"  half_window={cc['half_window']}  snp_only={cc['snp_only']}  attn_impl={cc['attn_impl']}")
print(f"  per-window norm   : mean {st['norms'].mean():.4g}  median {st['norms'].median():.4g}")
print(f"  delta norm (var.) : mean {st['delta_norms'][variant].mean():.4g}  "
      f"median {st['delta_norms'][variant].median():.4g}  max {st['delta_norms'].max():.4g}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(st["norms"].numpy(), bins=80, color="steelblue")
axes[0].set_title(f"{SELECT}: per-window embedding norm")
axes[0].set_xlabel("||emb||"); axes[0].set_ylabel("count")
axes[1].hist(st["delta_norms"][variant].numpy(), bins=80, color="indianred")
axes[1].set_title(f"{SELECT}: reference-delta norm (variant windows)")
axes[1].set_xlabel("||emb - emb_ref||"); axes[1].set_ylabel("count")
plt.tight_layout(); plt.show()

## 3. Cross-cache summary

One row per cache: shape, generation settings, and norm / reference-delta
summaries. `delta_*` are over variant windows only (reference rows are 0).

In [ ]:
rows = []
for name, cc in caches.items():
    st = STATS[name]
    variant = ~st["is_ref"]
    n_windows = len({(c, s, e) for (c, s, e, _) in cc["fps"]})
    rows.append({
        "cache": name,
        "half_window": cc["half_window"],
        "snp_only": cc["snp_only"],
        "attn_impl": cc["attn_impl"],
        "n_fps": cc["cache"].shape[0],
        "n_windows": n_windows,
        "D": cc["cache"].shape[1],
        "norm_mean": round(float(st["norms"].mean()), 4),
        "norm_median": round(float(st["norms"].median()), 4),
        "delta_mean": round(float(st["delta_norms"][variant].mean()), 5),
        "delta_median": round(float(st["delta_norms"][variant].median()), 5),
        "delta_max": round(float(st["delta_norms"].max()), 4),
        "frac_with_ref": round(float(st["has_ref"].float().mean()), 4),
    })
summary = pd.DataFrame(rows).sort_values(["half_window", "snp_only"]).reset_index(drop=True)
summary

## 4. Overlaid norm distributions

Compare the per-window norm and reference-delta distributions across all
loaded caches on shared axes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name in caches:
    st = STATS[name]
    variant = ~st["is_ref"]
    axes[0].hist(st["norms"].numpy(), bins=80, histtype="step", label=name, density=True)
    axes[1].hist(st["delta_norms"][variant].numpy(), bins=80, histtype="step",
                 label=name, density=True)
axes[0].set_title("per-window embedding norm"); axes[0].set_xlabel("||emb||")
axes[1].set_title("reference-delta norm (variant windows)")
axes[1].set_xlabel("||emb - emb_ref||")
for ax in axes:
    ax.set_ylabel("density"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Pairwise comparison on shared windows

For two caches that share fingerprints (same half-window, differing only by
`snp_only` or `attn_impl`), align rows by fingerprint and report the per-window
L2 diff — the same idea as `train_pipeline/compare_embeddings.py`. Caches with
different half-windows have disjoint windows, so `shared` will be ~empty.

In [ ]:
def compare_shared(name_a: str, name_b: str) -> dict:
    a, b = caches[name_a], caches[name_b]
    rows_a = {fp: a["cache"][i] for i, fp in enumerate(a["fps"])}
    rows_b = {fp: b["cache"][i] for i, fp in enumerate(b["fps"])}
    shared = rows_a.keys() & rows_b.keys()
    if not shared:
        print(f"{name_a} vs {name_b}: no shared fingerprints (different windowing?).")
        return {}
    diffs = torch.stack([(rows_a[k] - rows_b[k]).norm() for k in shared])
    base = torch.stack([rows_a[k].norm() for k in shared]).mean()
    out = {
        "shared": len(shared),
        "only_a": len(rows_a.keys() - rows_b.keys()),
        "only_b": len(rows_b.keys() - rows_a.keys()),
        "max_diff": float(diffs.max()),
        "mean_diff": float(diffs.mean()),
        "mean_norm": float(base),
        "mean_rel": float(diffs.mean() / base),
    }
    print(f"{name_a} vs {name_b}:")
    print(f"  shared {out['shared']:,}  (only A {out['only_a']:,}, only B {out['only_b']:,})")
    print(f"  max diff {out['max_diff']:.5g}  mean diff {out['mean_diff']:.5g}  "
          f"mean rel {out['mean_rel']:.3%}  (vs mean norm {out['mean_norm']:.4g})")
    return out

# Example: full-window vs snp-only at the same half-window.
_ = compare_shared("hw250", "hw250_snponly")